### Load data from pdf


In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('rag_paper.pdf')

# Load the document
data = loader.load()
print(data)

### Loading HTML files for RAG

In [ ]:
from langchain_community.document_loaders import UnstructuredHTMLLoader
# Create a document loader for unstructured HTML
loader = UnstructuredHTMLLoader('datacamp-blog.html')

# Load the document
data = loader.load()

# Print the first document's content
print(data[0].page_content)

# Print the first document's metadata
print(data[0].metadata)

### Getting started with text splitting

In [ ]:
text = '''RAG (retrieval augmented generation) is an advanced NLP model that combines retrieval mechanisms with generative capabilities. RAG aims to improve the accuracy and relevance of its outputs by grounding responses in precise, contextually appropriate data.'''

# Define a text splitter that splits on the '.' character
text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=70,  
    chunk_overlap=10  
)

# Split the text using text_splitter
chunks = text_splitter.split_text(text)
print(chunks)
print([len(chunk) for chunk in chunks])

### Recursively splitting documents

In [ ]:
loader = PyPDFLoader("rag_paper.pdf")
document = loader.load()

# Define a text splitter that splits recursively through the character list
text_splitter = RecursiveCharacterTextSplitter(
    separators=['\n', '.', ' ', ''],
    chunk_size=75,  
    chunk_overlap=10  
)

# Split the document using text_splitter
chunks = text_splitter.split_documents(document)
print(chunks)
print([len(chunk.page_content) for chunk in chunks])

### Embedding and storing documents

In [ ]:
# Initialize the OpenAI embedding model
embedding_model = OpenAIEmbeddings(api_key="<OPENAI_API_TOKEN>", model='text-embedding-3-small')

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### Initiating a RAG pipeline or a retriever

In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

### Creating a prompt template for RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Context: {context}
Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini", api_key="...", temperature=0)

### Building an LCEL retrieval chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Invoking the RAG chain for question answering

In [ ]:
result = chain.invoke({"question": "What are the key findings or results presented in the paper?"})
print(result)

In [ ]:
prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""
# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create an LCEL chain to test the prompt
chain = (
    prompt_template
    | llm
    
)

# Invoke the chain on the inputs provided
print(chain.invoke({"context": "DataCamp's RAG course was created by Meri Nova and James Chapman!", "question": "Who created DataCamp's RAG course?"}))

### Building the retrieval chain

Now for the finale of the chapter! You'll create a retrieval chain using LangChain's Expression Language (LCEL). This will combine the vector store containing your embedded document chunks from the RAG paper you loaded earlier, a prompt template, and an LLM so you can begin talking to your documents.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Context: {context}
Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini", api_key="...", temperature=0)

# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))

### Loading code files
Chatbots can not only access text files, but also code files like Python (.py) and Markdown files (.md). In this exercise, you'll load a Python file containing the RAG architecture you created in Chapter 1. Let's load the file to get a reminder!

In [ ]:
# Create a document loader for README.md and load it
loader = UnstructuredMarkdownLoader("README.md")

markdown_data = loader.load()
print(markdown_data[0])

In [ ]:
# Create a document loader for rag.py and load it
loader = PythonLoader("rag.py")

python_data = loader.load()
print(python_data[0])

### Splitting Python files
Although text and code files contain the same characters, code files contain structures beyond natural language. To retain this code-specific context during document splitting, you should program the splitter to first try to split on the most common code structure. Fortunately, LangChain provides functionality to do just that!

In [ ]:
# Create a Python-aware recursive character splitter
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=300, chunk_overlap=100
)

# Split the Python content into chunks
chunks = python_splitter.split_documents(python_data)

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n")

### Splitting by tokens
Splitting documents using RecursiveCharacterTextSplitter or CharacterTextSplitter is convenient, and can give you good performance in some cases, but it does have one drawback: they split using characters as base units, rather than tokens, which are processed by the model.

In [ ]:
# Get the encoding for gpt-4o-mini
encoding = tiktoken.encoding_for_model('gpt-4o-mini')

# Create a token text splitter
token_splitter = TokenTextSplitter(encoding_name=encoding.name, chunk_size=100, chunk_overlap=10)

# Split the PDF into chunks
chunks = token_splitter.split_documents(document)

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\nNo. tokens: {len(encoding.encode(chunk.page_content))}\n{chunk}\n")

### Splitting semantically
All of the splitting strategies you've used up to this point have the same drawback: the split doesn't consider the context of the surrounding text, so context can easily be lost during splitting.

In [ ]:
# Instantiate an OpenAI embeddings model
embedding_model = OpenAIEmbeddings(api_key="<OPENAI_API_TOKEN>", model='text-embedding-3-small')

# Create the semantic text splitter with desired parameters
semantic_splitter = SemanticChunker(
    embeddings=embedding_model, breakpoint_threshold_type="gradient", breakpoint_threshold_amount=0.8
)

# Split the document
chunks = semantic_splitter.split_documents(document)
print(chunks[0])

### BM25 in RAG

In [ ]:
retriever = BM25Retriever.from_documents(
    documents=chunks,
    k=5
)

chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt
         | llm
         | StrOutputParser()
)

### Understanding BM25
Before you start integrating a BM25 sparse retriever into your RAG architecture, it's best to test it on some short strings to get a intuition for how the retriever selects the documents.

You've been provided with three strings that you'll use as the basis for your BM25 retriever. The functionality required for this exercise is already loaded for you.

In [ ]:
chunks = [
    "RAG stands for Retrieval Augmented Generation.",
    "Graph Retrieval Augmented Generation uses graphs to store and utilize relationships between documents in the retrieval process.",
    "There are different types of RAG architectures; for example, Graph RAG."
]

# Initialize the BM25 retriever
bm25_retriever = BM25Retriever.from_texts(chunks, k=3)

# Invoke the retriever
results = bm25_retriever.invoke("Graph RAG")

# Extract the page content from the first result
print("Most Relevant Document:")
print(results[0].page_content)

### Sparse retrieval with BM25
Time to try out a sparse retrieval implementation! You'll create a BM25 retriever to ask questions about an academic paper on RAG, which has already been split into chunks called chunks. An OpenAI chat model and prompt have also been defined as llm and prompt, respectively. You can view the prompt provided by printing it in the console.

In [ ]:
# Create a BM25 retriever from chunks
retriever = BM25Retriever.from_documents(
    documents=chunks,
    k=5
)

# Create the LCEL retrieval chain
chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt
         | llm
         | StrOutputParser()
)

print(chain.invoke("What are knowledge-intensive tasks?"))

### Ragas context precision evaluation
To start your RAG evaluation journey, you'll begin by evaluating the context precision RAG metric using the ragas framework. Recall that context precision is essentially a measure of how relevant the retrieved documents are to the input query.

In [ ]:
from ragas.metrics import context_precision

# Define the context precision chain
context_precision_chain = EvaluatorChain(metric=context_precision, llm=llm, embeddings=embeddings)

# Evaluate the context precision of the RAG chain
eval_result = context_precision_chain({
  "question": "How does RAG enable AI applications?",
  "ground_truth": "RAG enables AI applications by integrating external data in generative models.",
  "contexts": [
    "RAG enables AI applications by integrating external data in generative models.",
    "RAG enables AI applications such as semantic search engines, recommendation systems, and context-aware chatbots."
  ]
})

print(f"Context Precision: {eval_result['context_precision']}")

### Ragas faithfulness evaluation
In this exercise, you'll evaluate the faithfulness of the RAG architecture you created at the end of Chapter 1. This chain has been re-defined for you and is available as through the variable chain.



In [ ]:
from ragas.metrics import faithfulness
from ragas.integrations.langchain import EvaluatorChain

# Query the retriever using the query and extract the document text
query = "How does RAG improve question answering with LLMs?"
retrieved_docs = [doc.page_content for doc in retriever.invoke(query)]

# Define the faithfulness chain
faithfulness_chain = EvaluatorChain(metric=faithfulness, llm=llm, embeddings=embeddings)

# Evaluate the faithfulness of the RAG chain
eval_result = faithfulness_chain({
  "question": query,
  "answer": chain.invoke(query),
  "contexts": retrieved_docs
})

print(eval_result)

### String evaluation
Time to really evaluate the final output by comparing it to an answer written by a subject matter expert. You'll use LangSmith's LangChainStringEvaluator class to perform this string comparison evaluation.

A prompt_template for string evaluation has already been written for you as:

`` You are an expert professor specialized in grading students' answers to questions.
You are grading the following question:{query}
Here is the real answer:{answer}
You are grading the following predicted answer:{result}
Respond with CORRECT or INCORRECT:
Grade: ``

The output from the RAG chain is stored as predicted_answer and the expert's response is stored as ref_answer.

All of the necessary classes have been imported for you.

Create the LangSmith QA string evaluator using the eval_llm and prompt_template provided.
Evaluate the RAG output, predicted_answer, by comparing it with the expert's response to the query, which is stored as ref_answer.

In [ ]:
# Create the QA string evaluator
qa_evaluator = LangChainStringEvaluator(
    "qa",
    config={
        "llm": eval_llm,
        "prompt": prompt_template
    }
)

query = "How does RAG improve question answering with LLMs?"

# Evaluate the RAG output by evaluating strings
score = qa_evaluator.evaluator.evaluate_strings(
    prediction=predicted_answer,
    reference=ref_answer,
    input=query
)

print(f"Score: {score}")

### Loading and chunking Wikipedia pages

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import TokenTextSplitter

raw_documents = WikipediaLoader(query="large language model").load()
text_splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=20)
documents = text_splitter.split_documents(raw_documents[:3])

print(documents[0])

### From text to Graph

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm = ChatOpenAI(api_key="...", temperature=0, model_name="gpt-4o-mini")
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(graph_documents)

### Creating graph documents
You've been provided with a document called famous_scientists containing a paragraph of text about famous scientists from the 20th century. In this exercise, you'll transform this unstructured text data into structured graph documents using LLMs!


Define an OpenAI chat LLM that will produce maximally deterministic responses.
Instantiate an LLM graph transformer for converting LangChain documents into graph documents.
Convert the text documents (docs) to graph documents using llm_transformer

In [ ]:
# Define the LLM
llm = ChatOpenAI(api_key="<OPENAI_API_TOKEN>", model="gpt-4o-mini", temperature=0)

# Instantiate the LLM graph transformer
llm_transformer = LLMGraphTransformer(llm=llm)

# Convert the text documents to graph documents
graph_documents = llm_transformer.convert_to_graph_documents(docs)
print(f"Derived Nodes:\n{graph_documents[0].nodes}\n")
print(f"Derived Edges:\n{graph_documents[0].relationships}")

### Building-up your graph database
So that you don't have to regenerate your graph documents every time, it's best practice to store them in a database that's specifically designed for graph data. Neo4j graph databases are an excellent choice for graph storage and retrieval, so you'll set one up using LangChain's Neo4j functionality.

Note: to use Neo4j in LangChain, you must also have the neo4j library installed as a dependency. In this course, this has already been done for you.


Instantiate the Neo4j graph using the url, user, and password variables provided.
Add the graph_documents to the graph, including their sources and additional entity labels.
Print the graph's schema.

In [ ]:
# Instantiate the Neo4j graph
graph = Neo4jGraph(url=url, username=user, password=password)

# Add the graph documents, sources, and include entity labels
graph.add_graph_documents(
    graph_documents,
    include_source=True,
    baseEntityLabel=True
)

graph.refresh_schema()

# Print the graph schema
print(graph.get_schema)

### Querying your graph database
Now that you have your database set up, it's time to begin querying. You'll view the graph schema to refamiliarize yourself with the nodes and relationships, and then write a Cypher query to query the graph.

If you need help with writing or fixing your Cypher query, you can use the ask_chatgpt(text: str) function that we've defined for you, which accepts a string argument.

The graph you created previously is available for you to use.


Query the graph using Cypher to return the person who is known for the Concept with id Theory Of Relativity; use the graph schema to identify the nodes and relationships.

In [ ]:
# Print the graph schema
print(graph.get_schema)

# Query the graph
results = graph.query("""
MATCH (relativity:Concept {id: "Theory Of Relativity"}) <-[:KNOWN_FOR]- (scientist)
RETURN scientist
""")

print(results[0])

### Chaining, Graph RAG style!
Now to bring everything together to create a Graph RAG QA chain! You've been provided with the same graph you've worked with throughout this chapter (with some potential variation in the specific nodes and relationships), and you'll connect this with another LLM to generate the Cypher query and return the natural language response.


Create a Graph Cypher QA chain using an OpenAI chat model and the graph you've created previously.
Invoke the chain with the input provided.
Extract and print the result text from the result.

In [ ]:
# Create the Graph Cypher QA chain
graph_qa_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(api_key="<OPENAI_API_TOKEN>", model="gpt-4o-mini", temperature=0), graph=graph, verbose=True
)

# Invoke the chain with the input provided
result = graph_qa_chain.invoke({"query": "Who discovered the element Radium?"})

# Print the result text
print(f"Final answer: {result['result']}")

### Graph RAG with filtering
For large and complex graphs, LLMs can sometimes struggle to accurately infer the most relevant nodes and relationships to build the Cypher query. Quite often, you will only need the LLM to be aware of a subset of the graph, and excluding particular node types will not only make it easier for the LLM to accurately create the Cypher query, but it will improve the query latency.

The graph database you've been working with is available as graph.

Create a graph QA chain that queries the graph database while ignoring nodes with the "Concept" type; an llm has been defined for you, and you should set verbose=True.
Invoke the graph_qa_chain with the input provided.

In [ ]:
# Create the graph QA chain excluding Concept
graph_qa_chain = GraphCypherQAChain.from_llm(
    graph=graph, llm=llm, exclude_types=["Concept"], verbose=True
)

# Invoke the chain with the input provided
result =graph_qa_chain.invoke ({"query": "Who was Marie Curie married to?"})
print(f"Final answer: {result['result']}")

> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: 'Marie Curie'})-[:SPOUSE]->(spouse:Person) RETURN spouse.id
Full Context:
[{'spouse.id': 'Pierre Curie'}]

> Finished chain.
Final answer: Marie Curie was married to Pierre Curie.

### Validating Cypher queries
When the LLMs generate the Cypher query, they have the graph schema available for reference; however, this doesn't mean there's absolute certainty that the query will reflect the schema perfectly. To improve reliability, you can validate and fix the generated query against the schema, which is particularly well-suited to fixing incorrect relationship directions.


Create a graph QA chain that queries the graph database, including an additional check to validate the generated Cypher query; an llm has been defined for you and you should set verbose=True.
Invoke the graph_qa_chain with the input provided.

In [ ]:
# Create the graph QA chain, validating the generated Cypher query
graph_qa_chain = GraphCypherQAChain.from_llm(
    graph=graph, llm=llm, validate_cypher=True, verbose=True
)

# Invoke the chain with the input provided
result = graph_qa_chain.invoke({"query": "Who won the Nobel Prize In Physics?"})
print(f"Final answer: {result['result']}")

Generated Cypher:
cypher
MATCH (p:Person)-[:AWARDED]->(a:Award {id: "Nobel Prize In Physics"})
RETURN p

Full Context:
[{'p': {'id': 'Albert Einstein'}}, {'p': {'id': 'Marie Curie'}}]

> Finished chain.
Final answer: Albert Einstein won the Nobel Prize in Physics.

### Creating a Cypher few-shot prompt
The final technique you'll utilize to improve the reliability of the generated Cypher is providing a few-shot prompt. Few-shot prompts are a great way of steering a model toward a desired output without needing to fine-tune it on a large dataset of examples.

A set of examples tailored to this particular use case is available as examples; feel free to print it in the shell to view its contents. You'll use these to create a few-shot prompt for the Cypher generation process. The graph you created before is still available as graph.

Create an example prompt template from the string provided.
Use the example_prompt, examples, and prefix and suffix provided to create a few-shot prompt.

In [ ]:
# Create an example prompt template
example_prompt = PromptTemplate.from_template(
    "User input: {question}\nCypher query: {query}"
)

# Create the few-shot prompt template
cypher_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="You are a Neo4j expert. Given an input question, create a syntactically correct Cypher query to run.\n\nHere is the schema information\n{schema}.\n\nBelow are a number of examples of questions and their corresponding Cypher queries.",
    suffix="User input: {question}\nCypher query: ",
    input_variables=["question"]
)

### Creating a Cypher few-shot prompt
The final technique you'll utilize to improve the reliability of the generated Cypher is providing a few-shot prompt. Few-shot prompts are a great way of steering a model toward a desired output without needing to fine-tune it on a large dataset of examples.

A set of examples tailored to this particular use case is available as examples; feel free to print it in the shell to view its contents. You'll use these to create a few-shot prompt for the Cypher generation process. The graph you created before is still available as graph.


Create the graph Cypher QA chain, replacing the default Cypher prompt with the few-shot prompt template; set verbose=True and validate the Cypher.
Invoke the graph_qa_chain with the input provided

In [ ]:
# Create an example prompt template
example_prompt = PromptTemplate.from_template(
    "User input: {question}\nCypher query: {query}"
)

# Create the few-shot prompt template
cypher_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="You are a Neo4j expert. Given an input question, create a syntactically correct Cypher query to run.\n\nHere is the schema information\n{schema}.\n\nBelow are a number of examples of questions and their corresponding Cypher queries.",
    suffix="User input: {question}\nCypher query: ",
    input_variables=["question"]
)

# Create the graph Cypher QA chain
graph_qa_chain = GraphCypherQAChain.from_llm(
    graph=graph, llm=llm, cypher_prompt=cypher_prompt, verbose=True, validate_cypher=True
)

# Invoke the chain with the input provided
result = graph_qa_chain.invoke({"query": "Which scientist proposed the Theory Of Relativity?"})
print(f"Final answer: {result['result']}")

Generated Cypher:
cypher
MATCH (p:Person)-[:KNOWN_FOR]->(c:Concept {id: 'Theory Of Relativity'}) RETURN p

Full Context:
[{'p': {'id': 'Albert Einstein'}}]

> Finished chain.
Final answer: Albert Einstein proposed the Theory of Relativity.